# Notebook 1 — The Network and Degree
**YTS+ DSEB 2026 · Project A · Harry Potter**

---

## The instrument

You know this story. Seven books, thousands of scenes, every character's arc.

This project does not teach you new facts about Harry Potter. It gives you a different instrument for reading what you already know — one that has never read a single page.

The instrument is a **co-occurrence network.** It sees only which characters Rowling chose to place near each other in the text, and how many times. That is all it sees. It cannot see who speaks to whom, who is brave, who drives the plot, who readers love or fear. It counts proximity.

When the machine's answer agrees with your reading: you have confirmed something about how Rowling *constructed* her story — not just how you *remember* it. Those are not always the same thing.

When it disagrees: the structure of the story and the impression it leaves on readers have come apart. That gap is not an error. **It is the finding.**

Every time the data surprises you, you face a choice:
- **Option A:** The algorithm measured something real that your reading missed. The story was written one way and felt another way.
- **Option B:** The algorithm measured a proxy that doesn't capture what actually matters — and you must explain what it missed.

Both are valid answers. But you cannot shrug. You must argue for one, with evidence from the books.

---

## Before any code — sealed prediction

Write on paper, independently:

> **If Harry Potter were removed from all 7 books entirely, which character becomes the most important?**

Seal it. Don't discuss it. You will open it at the end of Notebook 3.

---
## Setup

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import time
import os

os.makedirs('images', exist_ok=True)

DATA = 'data/'
BOOK_NAMES = ['Stone', 'Secrets', 'Azkaban', 'Fire', 'Phoenix', 'Prince', 'Hallows']

print('Ready.')

---
## Part 1 — What's in the data?

Before building anything, look at the raw data. Each row is a pair of characters who appear within 14 words of each other in the text. The `weight` is how many times that happened across Book 3.

Read the first 10 rows before running anything else. Ask yourself: **what kind of relationship does this edge capture?**

In [ ]:
df3 = pd.read_csv(DATA + 'hp_book3_edges.csv')

print(f'Rows (character pairs): {len(df3)}')
print(f'Columns: {df3.columns.tolist()}')
print()
df3.head(10)

In [ ]:
print(df3['weight'].describe())
print(f'\nMax weight pair:')
print(df3.sort_values('weight', ascending=False).head(1).to_string(index=False))

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df3['weight'], bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Co-appearance count (edge weight)')
plt.ylabel('Number of pairs')
plt.title('Book 3 — Distribution of co-appearance counts')
plt.tight_layout()
plt.savefig('images/book3_weight_hist.png', dpi=150)
plt.show()

print(f'Pairs appearing together only once: {(df3["weight"] == 1).sum()}')
print(f'Pairs appearing 10+ times: {(df3["weight"] >= 10).sum()}')
print(f'Pairs appearing 50+ times: {(df3["weight"] >= 50).sum()}')

In [ ]:
top_pairs = df3.sort_values('weight', ascending=False).head(10)
print('Top 10 most frequent pairs in Book 3:')
print(top_pairs.to_string(index=False))

**Crabbe appears with Goyle 95+ times.** Rowling placed them in the same moment in 95 scenes across one book.

Before you go further, think about two specific scenes you already know:

The tent scenes in Book 7 — Harry and Hermione alone for weeks. Rowling writes almost no names in those pages. It's *he said, she looked up, he sat across from her.* How many Harry-Hermione co-appearances does a 14-word name-counting window find there?

Now think about any scene where Draco arrives in the Hogwarts corridor with Crabbe and Goyle. Rowling writes: *Malfoy, Crabbe, and Goyle blocked the way.* Three names in one sentence.

**Write here — before running any more code:**

Which kind of scene does this algorithm count well? Which kind does it miss? What does that tell you about which characters will look more connected than they actually are?

*Write here.*

In [ ]:
t0 = time.time()

print(f'{"Book":<6} {"Name":<10} {"Pairs":>8} {"Characters":>12}')
print('-' * 40)

for i in range(1, 8):
    df = pd.read_csv(DATA + f'hp_book{i}_edges.csv')
    chars = set(df['source']) | set(df['target'])
    print(f'{i:<6} {BOOK_NAMES[i-1]:<10} {len(df):>8} {len(chars):>12}')

print(f'\nLoaded all 7 books in {time.time()-t0:.2f}s')

**The network does not grow monotonically.** Look at the character counts across books.

Before building the network, write your prediction for Part 4 (network expansion). You will check it after you plot the curves.

**Quick prediction — write here:**
- Which book has the most characters? ___
- Does the network ever shrink between books? Yes / No
- Which book shows the biggest single-book jump? ___

---
## Part 2 — Build the network

In [ ]:
t0 = time.time()

G3 = nx.from_pandas_edgelist(df3, 'source', 'target', edge_attr='weight')

print(f'Graph built in {time.time()-t0:.3f}s')
print(f'Characters (nodes): {G3.number_of_nodes()}')
print(f'Pairs who appear together (edges): {G3.number_of_edges()}')

In [ ]:
by_partners = sorted(G3.degree(), key=lambda x: x[1], reverse=True)

print('Top 5 by number of unique characters they appear with:')
for name, deg in by_partners[:5]:
    print(f'  {name}: {deg} unique partners')

print(f'\nAverage: {sum(d for _, d in G3.degree()) / G3.number_of_nodes():.1f} unique partners per character')

---
## Part 3 — Weighted degree

**Unweighted degree** counts how many unique characters you appear with.

**Weighted degree** sums up the weights — total co-appearances across all your partners.

Weighted degree tells you how much Rowling *used* this character across scenes — not just how many different people they appeared with.

**Before running the next cell — write 3 characters you expect in the top 5:**

1. 
2. 
3. 

In [ ]:
t0 = time.time()

degree3 = dict(G3.degree(weight='weight'))

print(f'Computed in {time.time()-t0:.4f}s')  # instant — note this for contrast later
print()

top15 = sorted(degree3.items(), key=lambda x: x[1], reverse=True)[:15]

print('Top 15 characters by weighted degree (Book 3):')
for i, (name, deg) in enumerate(top15):
    print(f'  {i+1:>2}. {name:<30} {deg}')

In [ ]:
names = [n for n, _ in top15]
values = [v for _, v in top15]

plt.figure(figsize=(9, 6))
plt.barh(names[::-1], values[::-1], color='steelblue')
plt.xlabel('Weighted degree (total co-appearances)')
plt.title('Book 3 — Top 15 characters by weighted degree')
plt.tight_layout()
plt.savefig('images/book3_degree.png', dpi=150)
plt.show()

**Check your predictions.**

Answer in writing — be specific:

> **Name one character who ranked higher than you expected. What does high weighted degree tell you about how Rowling wrote that character in Book 3 — and what does it NOT tell you?**

> **Name one character who ranked lower than you expected. What did you remember about them that the edge count doesn't capture?**

This is the first time the algorithm's answer and your reading might diverge. Both are measuring real things — but different things.

**Write here:**

*Higher than expected:*

*Lower than expected:*

---
## Part 4 — Network expansion across 7 books

You made a prediction above. Now compute it.

In [ ]:
t0 = time.time()

node_counts = []
edge_counts = []

for i in range(1, 8):
    df = pd.read_csv(DATA + f'hp_book{i}_edges.csv')
    G = nx.from_pandas_edgelist(df, 'source', 'target', edge_attr='weight')
    node_counts.append(G.number_of_nodes())
    edge_counts.append(G.number_of_edges())
    print(f'Book {i} ({BOOK_NAMES[i-1]}): {G.number_of_nodes()} characters, {G.number_of_edges()} pairs')

print(f'\nTotal time: {time.time()-t0:.2f}s')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = range(1, 8)

ax1.plot(x, node_counts, 'o-', color='steelblue', linewidth=2, markersize=8)
for i, n in enumerate(node_counts):
    ax1.annotate(str(n), (i+1, n), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax1.set_xticks(x)
ax1.set_xticklabels(BOOK_NAMES, rotation=30, ha='right')
ax1.set_ylabel('Number of characters')
ax1.set_title('Characters per book')
ax1.grid(alpha=0.3)

ax2.plot(x, edge_counts, 'o-', color='tomato', linewidth=2, markersize=8)
for i, n in enumerate(edge_counts):
    ax2.annotate(str(n), (i+1, n), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax2.set_xticks(x)
ax2.set_xticklabels(BOOK_NAMES, rotation=30, ha='right')
ax2.set_ylabel('Number of interacting pairs')
ax2.set_title('Pairs per book')
ax2.grid(alpha=0.3)

plt.suptitle("Rowling's world — per book", fontsize=13)
plt.tight_layout()
plt.savefig('images/network_expansion.png', dpi=150)
plt.show()

**Check your predictions. Then write one paragraph — not a description of the shape, but an explanation of the decisions.**

The curve is a trace of Rowling's choices about which characters to introduce, which to retire, and when to open or close the world. Your job is to name the decisions, not describe the numbers.

Specific things to address:
- The network **shrinks** from Book 1 to Book 3. You probably didn't predict that. Why does it shrink — what was Rowling doing in Books 2 and 3 that narrowed the world?
- The network **explodes** in Book 4. Name the specific narrative decision that caused this.
- The network **peaks** in Book 5. Then what happens — and why?

**Write your paragraph here:**

*Write here.*

---
## What to bring to Session 2

- Your sealed prediction (keep it sealed)
- Your answer to the edge question (Part 1): what three things does the edge NOT capture?
- Your expansion paragraph (Part 4)
- One thing the data showed you that you wouldn't have thought to look for

**Next:** Notebook 2 — Community Detection. The algorithm will partition the network into groups using only edge weights. No character names. No plot knowledge. We will find out: does it recover the factions Rowling built?